<a href="https://colab.research.google.com/github/KishoreKumar477/From-RNN-to-self-attention/blob/main/HowtoBuildRToT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets --quiet
from datasets import load_dataset

dataset = load_dataset('stanfordnlp/imdb')
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [ ]:
train_data = dataset['train']
test_data  = dataset['test']

print(train_data[0]['label'])   # 0 or 1?
print()
print(train_data[0]['text'][:300])  # first 300 characters


0

I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h


In [ ]:
import re

def tokenize(text):
    text = re.sub(r'<.*?>', ' ', text)        # remove HTML tags like <br/>
    text = re.sub(r'[^a-z0-9\s]', ' ', text.lower())  # lowercase, remove punctuation
    return text.split()

# Test it on the first review
tokens = tokenize(train_data[0]['text'])
print(tokens[:20])   # first 20 words
print(f'\nTotal words in this review: {len(tokens)}')

['i', 'rented', 'i', 'am', 'curious', 'yellow', 'from', 'my', 'video', 'store', 'because', 'of', 'all', 'the', 'controversy', 'that', 'surrounded', 'it', 'when', 'it']

Total words in this review: 291


In [ ]:
from collections import Counter

# Count every word across all 25,000 training reviews
counter = Counter()
for item in train_data:
    counter.update(tokenize(item['text']))

print(f'Total unique words found: {len(counter)}')
print(f'Most common 10 words: {counter.most_common(10)}')

Total unique words found: 74475
Most common 10 words: [('the', 336757), ('and', 164134), ('a', 163157), ('of', 145864), ('to', 135721), ('is', 107335), ('it', 96472), ('in', 93979), ('i', 87683), ('this', 76006)]


In [ ]:
MAX_VOCAB = 20000

# index 0 = <pad>  (for padding short reviews)
# index 1 = <unk>  (for words not in vocabulary)
word2idx = {'<pad>': 0, '<unk>': 1}
for word, count in counter.most_common(MAX_VOCAB):
    word2idx[word] = len(word2idx)

id2word = {idx: word for word, idx in word2idx.items()}



print(f'Vocabulary size: {len(word2idx)}. {len(id2word)}')
print(f'"movie" is index: {word2idx["movie"]}')
print(f'"the" is index:   {word2idx["the"]}')
print(f'"xyzabc" is index: {word2idx.get("xyzabc", 1)}')  # unknown word

Vocabulary size: 20002. 20002
"movie" is index: 18
"the" is index:   2
"xyzabc" is index: 1


In [ ]:
def encode(text):
    tokens = tokenize(text)[:256]
    return [word2idx.get(t, 1) for t in tokens]

def decode(coded):
    return [id2word.get(id,'<unk>') for id in coded]

print(train_data[0]['text'])
encoded = encode(train_data[0]['text'])
print(encoded[:20])
print(f'Length: {len(encoded)}')
print(f"decode: {(decode(encoded))}")

I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, eve

In [ ]:
import torch
import torch.nn as nn

def collate_batch(batch):
    labels = torch.tensor([item['label'] for item in batch], dtype=torch.float)
    texts  = [torch.tensor(encode(item['text']), dtype=torch.long) for item in batch]
    texts_padded = nn.utils.rnn.pad_sequence(texts, batch_first=True, padding_value=0)
    return labels, texts_padded

# Test on just 3 reviews
sample = [train_data[0], train_data[1], train_data[2]]
labels, texts = collate_batch(sample)
print(f'Labels shape: {labels.shape}')
print(f'Texts shape:  {texts.shape}')
print(f'Labels: {labels}')

Labels shape: torch.Size([3])
Texts shape:  torch.Size([3, 256])
Labels: tensor([0., 0., 0.])


In [ ]:
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}')

def collate_batch(batch):
    labels = torch.tensor([item['label'] for item in batch], dtype=torch.float)
    texts  = [torch.tensor(encode(item['text']), dtype=torch.long) for item in batch]
    texts_padded = nn.utils.rnn.pad_sequence(texts, batch_first=True, padding_value=0)
    return labels.to(device), texts_padded.to(device)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True,  collate_fn=collate_batch)
test_loader  = DataLoader(test_data,  batch_size=64, shuffle=False, collate_fn=collate_batch)

print(f'Train batches: {len(train_loader)}')
print(f'Test batches:  {len(test_loader)}')

Using: cuda
Train batches: 391
Test batches:  391


In [ ]:
import torch.nn as nn

embedding = nn.Embedding(num_embeddings=20002, embedding_dim=64, padding_idx=0)

print(embedding)
print(f'Embedding table size: {embedding.weight.shape}')

Embedding(20002, 64, padding_idx=0)
Embedding table size: torch.Size([20002, 64])


In [ ]:
rnn = nn.RNN(input_size=64, hidden_size=128, num_layers=2,
             batch_first=True, dropout=0.3)

print(rnn)

RNN(64, 128, num_layers=2, batch_first=True, dropout=0.3)


In [ ]:
fc = nn.Linear(128, 1)
print(fc)

Linear(in_features=128, out_features=1, bias=True)


In [ ]:
class SentimentRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(20002, 64, padding_idx=0)
        self.rnn       = nn.LSTM(64, 128, num_layers=2, batch_first=True, dropout=0.3)
        self.attention = nn.Linear(128,1) # V matrics
        self.fc        = nn.Linear(128, 1)
        self.dropout   = nn.Dropout(0.3)

    def forward(self, text):
        # 1.embed
        embedded        = self.dropout(self.embedding(text))
        # embedded: [64,256,64] (Batch,Sequence,Features)

        # 2. LSTM — but now keep ALL hidden states, not just the last
        output, (hidden,cell)  = self.rnn(embedded)
        # output: [64, 256, 128]  ← hidden state at every word

        # 3. compute attention score for every word
        scores = self.attention(output)         # [64, 256, 1]
        scores = torch.softmax(scores, dim=1)   # weights sum to 1

        # 4. weighted average of all hidden states
        final = (output * scores).sum(dim=1) #[64,128]

        # 5. classify
        return self.fc(self.dropout(final)).squeeze(1)

model = SentimentRNN().to(device)
print(model)

SentimentRNN(
  (embedding): Embedding(20002, 64, padding_idx=0)
  (rnn): LSTM(64, 128, num_layers=2, batch_first=True, dropout=0.3)
  (attention): Linear(in_features=128, out_features=1, bias=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)


##With Transformer

In [ ]:
import math

class SentimentTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding  = nn.Embedding(20002, 64, padding_idx=0)
        self.pos_encoding = nn.Embedding(256, 64)  # position embeddings
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=64,      # embedding size
                nhead=8,         # number of attention heads
                dim_feedforward=256,
                dropout=0.3,
                batch_first=True
            ),
            num_layers=2
        )
        self.fc      = nn.Linear(64, 1)
        self.dropout = nn.Dropout(0.3)

    def forward(self, text):
        # Step 1: word embeddings + position embeddings
        positions = torch.arange(text.size(1)).unsqueeze(0).to(device)
        embedded  = self.dropout(self.embedding(text) + self.pos_encoding(positions))
        # embedded: [64, 256, 64]

        # Step 2: self-attention across all words simultaneously
        output = self.transformer(embedded)
        # output: [64, 256, 64]

        # Step 3: average all word outputs
        final = output.mean(dim=1)
        # final: [64, 64]

        # Step 4: classify
        return self.fc(final).squeeze(1)

model = SentimentTransformer().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
print(model)

SentimentTransformer(
  (embedding): Embedding(20002, 64, padding_idx=0)
  (pos_encoding): Embedding(256, 64)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=256, bias=True)
        (dropout): Dropout(p=0.3, inplace=False)
        (linear2): Linear(in_features=256, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.3, inplace=False)
        (dropout2): Dropout(p=0.3, inplace=False)
      )
    )
  )
  (fc): Linear(in_features=64, out_features=1, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)


In [ ]:
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total:,}')

Trainable parameters: 1,396,545


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

def accuracy(preds, labels):
    rounded = torch.round(torch.sigmoid(preds))
    return (rounded == labels).float().mean()

print('Ready.')


Ready.


In [ ]:
def train_epoch(model, loader):
    model.train()
    total_loss, total_acc = 0, 0

    for labels, texts in loader:
        optimizer.zero_grad()        # imp.clear old gradients
        preds = model(texts)         # forward pass
        loss  = criterion(preds, labels)  # compute loss
        acc   = accuracy(preds, labels)
        loss.backward()              # backprop
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # clip exploding gradients
        optimizer.step()             # update weights

        total_loss += loss.item()
        total_acc  += acc.item()

    return total_loss / len(loader), total_acc / len(loader)

print('Ready.')

Ready.


In [ ]:
def evaluate(model, loader):
    model.eval()
    total_loss, total_acc = 0, 0

    with torch.no_grad():
        for labels, texts in loader:
            preds = model(texts)
            loss  = criterion(preds, labels)
            acc   = accuracy(preds, labels)
            total_loss += loss.item()
            total_acc  += acc.item()

    return total_loss / len(loader), total_acc / len(loader)

print('Ready.')

Ready.


In [ ]:
for epoch in range(1, 6):
    train_loss, train_acc = train_epoch(model, train_loader)
    val_loss,   val_acc   = evaluate(model, test_loader)
    print(f'Epoch {epoch} | Train Acc: {train_acc:.2%} | Val Acc: {val_acc:.2%}')

Epoch 1 | Train Acc: 88.25% | Val Acc: 85.95%
Epoch 2 | Train Acc: 89.29% | Val Acc: 86.15%
Epoch 3 | Train Acc: 90.24% | Val Acc: 86.42%
Epoch 4 | Train Acc: 91.01% | Val Acc: 86.61%
Epoch 5 | Train Acc: 91.14% | Val Acc: 86.79%


In [ ]:
def predict(sentence):
    model.eval()
    ids = torch.tensor(encode(sentence), dtype=torch.long).unsqueeze(0).to(device)
    with torch.no_grad():
        score = torch.sigmoid(model(ids)).item()
    label = 'POSITIVE 😊' if score >= 0.5 else 'NEGATIVE 😞'
    print(f'  "{sentence}"')
    print(f'  → {label}  (score: {score:.2%})\n')

predict("this movie was great")
predict("this movie was not great")
predict("i expected it to be terrible but it was actually great")
predict("started well but got worse and worse until the ending completely ruined it")

  "this movie was great"
  → POSITIVE 😊  (score: 99.97%)

  "this movie was not great"
  → POSITIVE 😊  (score: 84.53%)

  "i expected it to be terrible but it was actually great"
  → POSITIVE 😊  (score: 67.58%)

  "started well but got worse and worse until the ending completely ruined it"
  → NEGATIVE 😞  (score: 0.14%)

